# 🚀 Rocket Engine Safety Shutdown System
### Simple Reflex Agent + LLM Explanation (LangGraph + Groq + Gradio)

In [ ]:
!pip install langgraph langchain langchain-groq gradio

In [ ]:
import os
from typing import TypedDict

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

import gradio as gr

In [ ]:
# 🔑 Add your Groq API key here
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

## 🧠 Reflex Safety Logic (CRITICAL - Deterministic)

In [ ]:
THRESHOLD_TEMP = 1200

def reflex_shutdown_logic(temp: float):
    if temp > THRESHOLD_TEMP:
        return {
            "status": "SHUTDOWN",
            "reason": "Temperature exceeded safe threshold!",
            "action": "Engine shutdown triggered"
        }
    else:
        return {
            "status": "RUNNING",
            "reason": "Temperature within safe limits",
            "action": "Continue operation"
        }

## 🤖 LLM Explanation Layer (Non-Critical)

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

class AgentState(TypedDict):
    temperature: float
    reflex_output: dict
    llm_explanation: str

def llm_node(state: AgentState):
    prompt = f"""
    You are a rocket safety AI assistant.

    Temperature: {state['temperature']}
    System Decision: {state['reflex_output']['status']}
    Reason: {state['reflex_output']['reason']}

    Explain this decision professionally.
    """

    response = llm.invoke(prompt)

    return {
        "llm_explanation": response.content
    }

## 🔗 LangGraph Workflow

In [ ]:
def reflex_node(state: AgentState):
    result = reflex_shutdown_logic(state["temperature"])
    return {"reflex_output": result}

builder = StateGraph(AgentState)

builder.add_node("reflex", reflex_node)
builder.add_node("llm", llm_node)

builder.set_entry_point("reflex")
builder.add_edge("reflex", "llm")
builder.add_edge("llm", END)

graph = builder.compile()

## 🎨 Gradio UI

In [ ]:
def run_system(temp):
    result = graph.invoke({"temperature": temp})

    return (
        result["reflex_output"]["status"],
        result["reflex_output"]["reason"],
        result["reflex_output"]["action"],
        result["llm_explanation"]
    )

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.HTML("""
    <div style="text-align:center;">
        <h1>🚀 Rocket Engine Safety Shutdown</h1>
        <h3>Collins Aerospace</h3>
        <marquee behavior="scroll" direction="left" style="color:red; font-weight:bold;">
            Rocket Engine Safety Shutdown System • Real-time Monitoring • Fail-safe Reflex Architecture
        </marquee>
    </div>
    """)

    temp_input = gr.Slider(0, 2000, value=800, label="Engine Temperature (°C)")

    run_btn = gr.Button("Run Safety Check")

    status = gr.Textbox(label="System Status")
    reason = gr.Textbox(label="Reason")
    action = gr.Textbox(label="Action Taken")
    explanation = gr.Textbox(label="AI Explanation")

    run_btn.click(
        fn=run_system,
        inputs=temp_input,
        outputs=[status, reason, action, explanation]
    )

demo.launch()